# Lecture 6 — Class Exercise
## Choropleth Maps

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:
# Task 1 — Life expectancy relative to the world average (2007)
# Chart: px.choropleth (country polygons, whole-world view -> a projection is the right fit, no street zoom needed)
# Scale: DIVERGING (RdBu), because the data has a meaningful midpoint = 0 (the global average):
#        red = below average, blue = above average.
gm = px.data.gapminder()
gm_2007 = gm[gm["year"] == 2007].copy()

world_avg = gm_2007["lifeExp"].mean()
gm_2007["deviation"] = (gm_2007["lifeExp"] - world_avg).round(2)

# Which continent is furthest below average?
by_cont = gm_2007.groupby("continent")["deviation"].mean().sort_values()
worst = by_cont.index[0]
print(f"World average: {world_avg:.1f} years")
print(by_cont.round(1))

lim = gm_2007["deviation"].abs().max()          # symmetric range so white = exactly 0
fig1 = px.choropleth(
    gm_2007,
    locations="iso_alpha",
    color="deviation",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    range_color=(-lim, lim),
    hover_name="country",
    hover_data={"iso_alpha": False, "lifeExp": ":.1f", "deviation": ":+.1f"},
    labels={"lifeExp": "Life expectancy (years)", "deviation": "Years vs world average"},
    title=f"{worst} is furthest below the {world_avg:.0f}-year world average in life expectancy "
          f"({by_cont.iloc[0]:+.1f} years on average, 2007)",
)
fig1.update_layout(margin=dict(l=0, r=0, t=70, b=0), height=600,
                   coloraxis_colorbar_title="Years vs<br>world avg")
fig1.show()


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** geoBoundaries (open licence), Iran ADM1 = the 31 provinces. Downloaded automatically in the code cell from `https://www.geoboundaries.org/api/current/gbOpen/IRN/ADM1/`.

**Chart type chosen** (`px.choropleth` or `px.choropleth_map`) **and reason:**

`px.choropleth_map`. It draws on a real tile basemap, so the map can be **zoomed and panned** (mouse wheel / drag), which helps to inspect small provinces such as Alborz, Qom and Tehran. `px.choropleth` uses a fixed projection without street-level zoom.

**Colour scale chosen** (sequential or diverging) **and reason:**

Sequential (`YlOrRd`). The variable is population (2016 census): it goes from low to high with no meaningful midpoint, so a diverging scale would be misleading.


In [ ]:
# Task 2 — Iran provinces, population (2016 census, rounded figures)
import json, re, difflib, requests
import pandas as pd
import plotly.express as px

# 1) Download the GeoJSON (31 provinces)
meta = requests.get("https://www.geoboundaries.org/api/current/gbOpen/IRN/ADM1/", timeout=60).json()
geo_url = meta.get("simplifiedGeometryGeoJSON") or meta["gjDownloadURL"]
iran_geojson = requests.get(geo_url, timeout=120).json()
with open("iran_provinces.geojson", "w") as fh:      # local copy, push it to GitHub too
    json.dump(iran_geojson, fh)

# 2) Inspect properties -> featureidkey
print(iran_geojson["features"][0]["properties"])
# province name is in 'shapeName'  ->  featureidkey = "properties.shapeName"

# 3) Data: one numeric value per province
pop = {
 "Tehran":13267637,"Razavi Khorasan":6434501,"Isfahan":5120850,"Fars":4851274,"Khuzestan":4710509,
 "East Azerbaijan":3909652,"Mazandaran":3283582,"West Azerbaijan":3265219,"Kerman":3164718,
 "Sistan and Baluchestan":2775014,"Alborz":2712400,"Gilan":2530696,"Kermanshah":1952434,
 "Golestan":1868819,"Lorestan":1760649,"Hamadan":1738234,"Hormozgan":1776415,"Kurdistan":1603011,
 "Markazi":1429475,"Qazvin":1273761,"Ardabil":1270420,"Qom":1292283,"Yazd":1138533,"Bushehr":1163400,
 "Zanjan":1057461,"Chaharmahal and Bakhtiari":947763,"North Khorasan":863092,"South Khorasan":768898,
 "Kohgiluyeh and Boyer-Ahmad":713052,"Semnan":702360,"Ilam":580158,
}
df = pd.DataFrame({"province": list(pop), "population": list(pop.values())})

# 4) Match my names to the GeoJSON names (spelling differs between sources)
def norm(s): return re.sub(r"[^a-z]", "", s.lower().replace("province", ""))
geo_names = [f["properties"]["shapeName"] for f in iran_geojson["features"]]
lookup = {norm(n): n for n in geo_names}
def match(name):
    k = norm(name)
    if k in lookup: return lookup[k]
    best = difflib.get_close_matches(k, lookup.keys(), n=1, cutoff=0.6)
    return lookup[best[0]] if best else None
df["geo_name"] = df["province"].apply(match)
print("Unmatched in my data:", df[df.geo_name.isna()].province.tolist())
print("GeoJSON names not used:", sorted(set(geo_names) - set(df.geo_name.dropna())))
df = df.dropna(subset=["geo_name"])
df["population_m"] = (df["population"] / 1e6).round(2)

# 5) Zoomable choropleth
fig = px.choropleth_map(
    df,
    geojson=iran_geojson,
    locations="geo_name",
    featureidkey="properties.shapeName",
    color="population",
    color_continuous_scale="YlOrRd",
    hover_name="province",
    hover_data={"geo_name": False, "population": ":,", "population_m": True},
    map_style="carto-positron",
    center={"lat": 32.4, "lon": 53.7},
    zoom=4.2,
    opacity=0.8,
    labels={"population": "Population", "population_m": "Millions"},
    title="Iran's people cluster in the north and centre: Tehran alone holds about 13 million",
)
fig.update_layout(margin=dict(l=0, r=0, t=60, b=0), height=700)
fig.show()


In [ ]:
# Export so the map can be seen on GitHub (GitHub does NOT run Plotly's JavaScript in notebooks)
fig.write_html("iran_population_map.html", include_plotlyjs="cdn")   # fully zoomable version

try:   # static preview that GitHub shows inside the notebook (pip install kaleido)
    fig.write_image("iran_population_map.png", width=1100, height=700, scale=2)
    from IPython.display import Image, display
    display(Image("iran_population_map.png"))
except Exception as e:
    print("PNG export skipped:", e)
